In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition # toolNode and toolCondition
from langgraph.checkpoint.memory import InMemorySaver # checkpointers
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv

import sys

load_dotenv()


True

In [2]:
class Messages(TypedDict):
    messages: Annotated[BaseMessage, add_messages]

chatModel = None

In [3]:

MCP_CONFIG = {
    "math": {
        "transport": "stdio",
        "command": sys.executable,   # the venv python running this notebook
        "args": ["/Users/sriharshamadireddy/Desktop/Personal/Learning/DataScience/genai/langgraph/14. Langgraph with MCP/localMCPServer.py"],
    }
}

mcp_client = MultiServerMCPClient(MCP_CONFIG)

async def _initiateChatModel(): 
    tools = await mcp_client.get_tools()
    print(tools)
    chatModelOld = ChatOpenAI();
    chatModel = chatModelOld.bind_tools(tools);
    return chatModel

async def _initiateToolNode():
    tools = await mcp_client.get_tools()
    return ToolNode(tools)



In [4]:
async def chatNode(state: Messages)-> Messages:
    query = state['messages']
    lastQuery = query[-1]

    chatModel = await _initiateChatModel()
    response = chatModel.invoke(query)
    return {'messages': [response]}


toolNode = await _initiateToolNode()
    

In [5]:
graph = StateGraph(Messages)

graph.add_node('chat_node', chatNode)
graph.add_node('tools', toolNode)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')


workflow = graph.compile()

In [6]:
response = await workflow.ainvoke({'messages': ["result of add: 1 , 2"]})
response

[StructuredTool(name='add', description='Use this tool if you want to add 2 numbers ', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x10fe6fd80>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x106a5efc0>), StructuredTool(name='minus', description='Use this tool if you want to subtract 2 numbers ', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x10fe6fd80>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x1180179c0>), StructuredTool(n

{'messages': [HumanMessage(content='result of add: 1 , 2', additional_kwargs={}, response_metadata={}, id='890ef278-0645-488a-b086-8329c84ce625'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 152, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EIbnbErwMxM3VT0gY6wDLKjTPKPc8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0534c-5fb9-71f0-98fc-4867a546c77a-0', tool_calls=[{'name': 'add', 'args': {'a': 1, 'b': 2}, 'id': 'call_cMm1Xgqt2Dx0BgDg8JDU17SM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_t